# YOLO11 Helmet Detection — Complete Colab Notebook

This notebook takes you from the dataset ZIP file to training, evaluation, graphs, and testing a new image.

**Before running:**
1. In Google Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run the cells from top to bottom.
3. When prompted, upload `helmet.v1i.yolov11.zip` directly from your computer.
4. Sign in to Google Drive when asked. Training outputs will be saved there.

Your dataset has one class, `helmet`. This model detects helmets; it does not independently classify people as wearing or not wearing helmets.

## 1. Check GPU and install YOLO

In [ ]:
import torch, os, sys, glob, shutil, zipfile, yaml, random
from pathlib import Path

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. In Colab, select Runtime > Change runtime type > GPU.")

!pip -q install -U ultralytics

import ultralytics
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__version__)

## 2. Upload the dataset ZIP
If the ZIP is already in `/content`, the cell will use it. Otherwise, it opens Colab's file picker.

In [ ]:
ZIP_PATH = "/content/helmet.v1i.yolov11.zip"

if not os.path.exists(ZIP_PATH):
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    uploaded_name = next(iter(uploaded))
    ZIP_PATH = os.path.join("/content", uploaded_name)

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"ZIP file not found: {ZIP_PATH}")

print("Using ZIP:", ZIP_PATH)
print("ZIP size (MB):", round(os.path.getsize(ZIP_PATH) / (1024**2), 2))

## 3. Extract the ZIP

In [ ]:
EXTRACT_DIR = "/content/helmet_dataset"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print("Top-level contents:", os.listdir(EXTRACT_DIR)[:30])

## 4. Find and inspect `data.yaml`

In [ ]:
yaml_candidates = list(Path(EXTRACT_DIR).rglob("data.yaml"))
if not yaml_candidates:
    yaml_candidates = list(Path(EXTRACT_DIR).rglob("*.yaml"))

if not yaml_candidates:
    raise FileNotFoundError("No YAML dataset configuration was found inside the ZIP.")

DATA_YAML_ORIGINAL = str(yaml_candidates[0])
print("Configuration file:", DATA_YAML_ORIGINAL)

with open(DATA_YAML_ORIGINAL, "r") as f:
    original_config = yaml.safe_load(f)

print("\nOriginal YAML contents:")
print(yaml.safe_dump(original_config, sort_keys=False))

## 5. Resolve dataset folders and write a Colab-ready YAML
This cell checks the extracted folders instead of assuming their exact nesting. It expects image folders such as `train/images`, `valid/images` (or `val/images`), and optionally `test/images`.

In [ ]:
def find_split_images(split_names):
    found = []
    for p in Path(EXTRACT_DIR).rglob("images"):
        parent_name = p.parent.name.lower()
        if parent_name in split_names and p.is_dir():
            found.append(p)
    if not found:
        return None
    # Prefer the shallowest matching path
    return sorted(found, key=lambda p: len(p.parts))[0]

train_dir = find_split_images({"train"})
val_dir = find_split_images({"valid", "val"})
test_dir = find_split_images({"test"})

if train_dir is None or val_dir is None:
    raise FileNotFoundError(
        "Could not find both train/images and valid/images (or val/images). "
        "Inspect the extracted folders above and adjust the paths manually."
    )

# Build the dataset root as the parent of the split folders (e.g. .../train/images -> ...)
split_roots = [train_dir.parent.parent, val_dir.parent.parent]
if test_dir is not None:
    split_roots.append(test_dir.parent.parent)

# Normally these all point to the same dataset root. Use the common parent.
dataset_root = Path(os.path.commonpath([str(p) for p in split_roots]))

# Class names come from the dataset YAML.
names = original_config.get("names", {0: "helmet"})
nc = original_config.get("nc", len(names) if isinstance(names, (list, dict)) else 1)

# Normalize class names into a YAML-friendly structure.
if isinstance(names, dict):
    names = {int(k): v for k, v in names.items()}
elif isinstance(names, list):
    names = {i: name for i, name in enumerate(names)}

config_fixed = {
    "path": str(dataset_root),
    "train": str(train_dir.relative_to(dataset_root)),
    "val": str(val_dir.relative_to(dataset_root)),
    "names": names,
    "nc": int(nc)
}
if test_dir is not None:
    config_fixed["test"] = str(test_dir.relative_to(dataset_root))

FIXED_YAML = "/content/helmet_data_colab.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.safe_dump(config_fixed, f, sort_keys=False)

print("Dataset root:", dataset_root)
print("Train images:", train_dir)
print("Validation images:", val_dir)
print("Test images:", test_dir)
print("\nCorrected YAML:")
print(yaml.safe_dump(config_fixed, sort_keys=False))

## 6. Count images and check labels

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    return sum(1 for p in Path(folder).iterdir()
               if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

for label, folder in [("Train", train_dir), ("Validation", val_dir), ("Test", test_dir)]:
    if folder is not None:
        print(f"{label} images: {count_images(folder)}")

# YOLO labels are normally in a sibling labels folder.
for split_dir in [train_dir, val_dir, test_dir]:
    if split_dir is not None:
        labels_dir = split_dir.parent / "labels"
        print(f"{split_dir.parent.name}: labels folder exists = {labels_dir.is_dir()}")
        if labels_dir.is_dir():
            print("  label files:", len(list(labels_dir.glob("*.txt"))))

## 7. Connect Google Drive and create an output folder

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/YOLO_Helmet_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Training outputs will be saved in:", PROJECT_DIR)

## 8. Load pretrained YOLO11n

In [ ]:
model = YOLO("yolo11n.pt")
print("Pretrained YOLO11n loaded.")

## 9. Train the model

Default settings:
- 50 epochs
- 640-pixel image size
- batch size 16
- GPU 0 (Colab T4)

If you get a CUDA out-of-memory error, change `BATCH_SIZE` to `8` or `4`, then rerun this cell. Training results are saved to Google Drive.

In [ ]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16

train_results = model.train(
    data=FIXED_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else "cpu",
    project=PROJECT_DIR,
    name="helmet_yolo11n_50epochs",
    exist_ok=True,
    save=True,
    plots=True,
    workers=2,
    seed=42
)

RUN_DIR = os.path.join(PROJECT_DIR, "helmet_yolo11n_50epochs")
BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")
LAST_WEIGHTS = os.path.join(RUN_DIR, "weights", "last.pt")

print("Training finished.")
print("Run folder:", RUN_DIR)
print("best.pt exists:", os.path.isfile(BEST_WEIGHTS), BEST_WEIGHTS)
print("last.pt exists:", os.path.isfile(LAST_WEIGHTS), LAST_WEIGHTS)

## 10. Load the best model
This cell gives a clear error if training did not finish or `best.pt` was not created.

In [ ]:
if not os.path.isfile(BEST_WEIGHTS):
    raise FileNotFoundError(
        "best.pt was not found. Check the training output above for an error "
        "or inspect the run folder in Google Drive."
    )

best_model = YOLO(BEST_WEIGHTS)
print("Loaded best model:", BEST_WEIGHTS)

## 11. Evaluate the model

If the dataset includes a test split, this evaluates on that split. Otherwise, it evaluates on validation data and prints a reminder. Do not describe validation results as test results.

In [ ]:
has_test_split = "test" in config_fixed and test_dir is not None

if has_test_split:
    print("Evaluating on the TEST split.")
    metrics = best_model.val(data=FIXED_YAML, split="test",
                             device=0 if torch.cuda.is_available() else "cpu",
                             plots=True, project=PROJECT_DIR,
                             name="helmet_test_evaluation", exist_ok=True)
    evaluation_split = "test"
else:
    print("No test split found. Evaluating on the VALIDATION split instead.")
    metrics = best_model.val(data=FIXED_YAML, split="val",
                             device=0 if torch.cuda.is_available() else "cpu",
                             plots=True, project=PROJECT_DIR,
                             name="helmet_validation_evaluation", exist_ok=True)
    evaluation_split = "validation"

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

print("\nEVALUATION RESULTS —", evaluation_split.upper())
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"mAP50:      {map50:.4f}")
print(f"mAP50-95:   {map50_95:.4f}")

## 12. Display training and evaluation graphs

In [ ]:
from IPython.display import display, Image

# Display PNG files from the training run and evaluation folders
folders_to_check = [RUN_DIR,
                    os.path.join(PROJECT_DIR, "helmet_test_evaluation"),
                    os.path.join(PROJECT_DIR, "helmet_validation_evaluation")]

shown = set()
for folder in folders_to_check:
    if os.path.isdir(folder):
        for graph in sorted(glob.glob(os.path.join(folder, "*.png"))):
            if graph not in shown:
                shown.add(graph)
                print(os.path.basename(graph), "—", folder)
                display(Image(filename=graph))
if not shown:
    print("No PNG graphs found in the expected folders. Check the run directory.")

## 13. View the training results table (CSV)

In [ ]:
results_csv = os.path.join(RUN_DIR, "results.csv")
if os.path.isfile(results_csv):
    import pandas as pd
    training_table = pd.read_csv(results_csv)
    display(training_table.tail())
else:
    print("results.csv not found at:", results_csv)

## 14. Upload a new image and run helmet detection

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import cv2

print("Choose a test image from your computer.")
uploaded_test = files.upload()

if uploaded_test:
    test_image_path = os.path.join("/content", next(iter(uploaded_test)))
    predictions = best_model.predict(
        source=test_image_path,
        conf=0.25,
        device=0 if torch.cuda.is_available() else "cpu",
        save=True
    )

    plotted = predictions[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("YOLO11 Helmet Detection")
    plt.show()
else:
    print("No image uploaded.")

## 15. Final file locations and optional download

In [ ]:
print("BEST MODEL:", BEST_WEIGHTS)
print("LAST MODEL:", LAST_WEIGHTS)
print("TRAINING RESULTS:", RUN_DIR)

print("\nFiles saved in the training run:")
for root, dirs, files_in_folder in os.walk(RUN_DIR):
    for filename in files_in_folder:
        print(os.path.join(root, filename))

## Important notes
- `best.pt` is created only after training successfully completes. It will not exist before Step 9 finishes.
- `best.pt` is the best checkpoint selected during training; `last.pt` is the final checkpoint.
- Precision, recall, mAP50 and mAP50-95 are object-detection metrics. They are not ordinary classification accuracy.
- Keep the exact metrics printed by the evaluation cell for your report; do not invent or round them inaccurately.
- Your one-class dataset detects the `helmet` object only. It cannot determine helmet compliance unless the dataset is redesigned with suitable classes/annotations.
- If Colab disconnects during training, the training process can stop. Completed outputs saved in Drive remain there.